In [1]:
import geemap
import ee

In [2]:
Map = geemap.Map() #center参数是打开地图的中心点经纬度坐标
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position'], widget=HBox(children=(ToggleButton(value=…

In [ ]:
start_time = '2020-01' # 开始时间
end_time = '2021-01'   # 结束时间

In [ ]:
roi = geemap.geojson_to_ee("E:/Data/GIS/JX.json")
Map.centerObject(roi)
Map.addLayer(roi, {}, "roi")

In [ ]:
def fun(img):
    return img.clip(roi.geometry()).select("sur_refl_b01", "sur_refl_b02")
def fun1(img):
    return img.addBands(img.expression(
                '(NIR-RED)/(NIR+RED)', {
                  'NIR': img.select('sur_refl_b02'),
                  'RED': img.select('sur_refl_b01')
            }).select([0], ["NDVI"])).select("NDVI")

In [ ]:
modis = ee.ImageCollection("MODIS/006/MOD09Q1")\
        .filterDate(start_time, end_time)\
        .filterBounds(roi.geometry())\
        .map(fun).map(fun1)

In [ ]:
def fun2(img, first):
    return ee.Image(first).addBands(img)

In [ ]:
modis_img = ee.Image(modis.iterate(fun2, ee.Image())).select("NDV.*?")

In [ ]:
name = modis_img.bandNames().getInfo()

In [ ]:
new_name = []
for i in range(1, 366, 8):
    new_name.append(str(i))

In [ ]:
image = modis_img.select(name, new_name)

In [ ]:
image.bandNames().getInfo()

In [ ]:
point = geemap.shp_to_ee("E:/Data/GIS/points.shp")

In [ ]:
csv = "E:/public/out_points.shp"
# geemap.extract_values_to_points(point, image, csv, geometries=True)

In [ ]:
landsat8 = ee.ImageCollection("LANDSAT/LC08/C01/T1")\
        .filterDate(start_time, end_time)\
        .filterBounds(roi.geometry())

In [ ]:
landsat8.aggregate_array("system:id").getInfo()

In [ ]:
landsat8.size().getInfo()

In [ ]:
point.first().getInfo()

In [ ]:
geo = point.geometry().getInfo()

In [ ]:
geo

In [3]:
import pandas as pd
shp = "E:/public/out_points.shp"
points = geemap.shp_to_ee(shp)

In [8]:
df = geemap.ee_to_pandas(points)
df

,system:index,Field1,NDVI01_a,NDVI01_b,NDVI02_a,NDVI02_b,NDVI03_a,NDVI03_b,NDVI04_a,NDVI04_b,...,NDVI09_a,NDVI09_b,NDVI10_a,NDVI10_b,NDVI11_a,NDVI11_b,NDVI12_a,NDVI12_b,X,Y
0,0,0,0.2072,0.2237,0.2510,0.2677,0.2674,0.2527,0.1888,0.1656,...,0.3015,0.2851,0.2888,0.2934,0.2869,0.2661,0.2222,0.2211,115.944450,28.696499
1,1,1,0.7611,0.7252,0.6900,0.6767,0.6682,0.6885,0.7234,0.7534,...,0.8333,0.8171,0.7883,0.7604,0.7572,0.7588,0.7712,0.7751,114.523780,26.624906
2,2,2,-0.2352,-0.2393,-0.1888,-0.1612,-0.0635,-0.0197,0.0537,0.0455,...,0.1002,0.1068,0.0982,0.0070,-0.0563,-0.1254,-0.1821,-0.2167,114.399671,29.053060
3,3,3,0.7180,0.7039,0.6884,0.6876,0.6930,0.7016,0.7435,0.7749,...,0.8225,0.8184,0.8113,0.8017,0.7900,0.7740,0.7500,0.7473,114.373410,28.859427
4,4,4,0.7371,0.7168,0.6988,0.6934,0.6997,0.7080,0.7444,0.7724,...,0.8517,0.8493,0.8417,0.8307,0.8177,0.7980,0.7858,0.7710,115.861399,25.148718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4969,4969,4969,0.2655,0.3314,0.3866,0.3764,0.3455,0.3283,0.3006,0.3220,...,0.5776,0.5754,0.5427,0.4459,0.3499,0.2420,0.1844,0.1870,116.738013,29.075583
4970,4970,4970,0.6798,0.7060,0.7408,0.7611,0.7618,0.7514,0.7512,0.7507,...,0.8204,0.8227,0.8154,0.7901,0.7586,0.7173,0.6824,0.6716,116.877634,29.295952
4971,4971,4971,-0.2130,-0.2516,-0.2397,-0.1901,-0.1803,-0.1245,0.0182,0.0890,...,-0.1943,-0.1427,-0.1339,-0.1393,-0.1551,-0.1589,-0.1846,-0.2019,116.323362,28.661318
4972,4972,4972,0.6863,0.5512,0.4851,0.3357,0.4433,0.4233,0.6707,0.7768,...,0.8095,0.7683,0.6905,0.6520,0.6676,0.7246,0.7619,0.7411,117.726469,28.687242


In [9]:
df1 = df.iloc[:, -2:]
df1

,X,Y
0,115.944450,28.696499
1,114.523780,26.624906
2,114.399671,29.053060
3,114.373410,28.859427
4,115.861399,25.148718
...,...,...
4969,116.738013,29.075583
4970,116.877634,29.295952
4971,116.323362,28.661318
4972,117.726469,28.687242


In [10]:
df2 = df.iloc[:, 2:-2]
df2

,NDVI01_a,NDVI01_b,NDVI02_a,NDVI02_b,NDVI03_a,NDVI03_b,NDVI04_a,NDVI04_b,NDVI05_a,NDVI05_b,...,NDVI08_a,NDVI08_b,NDVI09_a,NDVI09_b,NDVI10_a,NDVI10_b,NDVI11_a,NDVI11_b,NDVI12_a,NDVI12_b
0,0.2072,0.2237,0.2510,0.2677,0.2674,0.2527,0.1888,0.1656,0.1460,0.1860,...,0.3762,0.3378,0.3015,0.2851,0.2888,0.2934,0.2869,0.2661,0.2222,0.2211
1,0.7611,0.7252,0.6900,0.6767,0.6682,0.6885,0.7234,0.7534,0.8171,0.8284,...,0.8390,0.8538,0.8333,0.8171,0.7883,0.7604,0.7572,0.7588,0.7712,0.7751
2,-0.2352,-0.2393,-0.1888,-0.1612,-0.0635,-0.0197,0.0537,0.0455,0.0039,-0.0361,...,0.0046,0.0460,0.1002,0.1068,0.0982,0.0070,-0.0563,-0.1254,-0.1821,-0.2167
3,0.7180,0.7039,0.6884,0.6876,0.6930,0.7016,0.7435,0.7749,0.8087,0.8374,...,0.8287,0.8274,0.8225,0.8184,0.8113,0.8017,0.7900,0.7740,0.7500,0.7473
4,0.7371,0.7168,0.6988,0.6934,0.6997,0.7080,0.7444,0.7724,0.8144,0.8286,...,0.8350,0.8435,0.8517,0.8493,0.8417,0.8307,0.8177,0.7980,0.7858,0.7710
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4969,0.2655,0.3314,0.3866,0.3764,0.3455,0.3283,0.3006,0.3220,0.4548,0.5592,...,0.6015,0.5811,0.5776,0.5754,0.5427,0.4459,0.3499,0.2420,0.1844,0.1870
4970,0.6798,0.7060,0.7408,0.7611,0.7618,0.7514,0.7512,0.7507,0.7721,0.7970,...,0.8266,0.8187,0.8204,0.8227,0.8154,0.7901,0.7586,0.7173,0.6824,0.6716
4971,-0.2130,-0.2516,-0.2397,-0.1901,-0.1803,-0.1245,0.0182,0.0890,0.1747,0.1407,...,-0.3061,-0.2604,-0.1943,-0.1427,-0.1339,-0.1393,-0.1551,-0.1589,-0.1846,-0.2019
4972,0.6863,0.5512,0.4851,0.3357,0.4433,0.4233,0.6707,0.7768,0.8428,0.8368,...,0.8044,0.8067,0.8095,0.7683,0.6905,0.6520,0.6676,0.7246,0.7619,0.7411


In [11]:
df3 = pd.concat([df1, df2], axis=1)
df3

,X,Y,NDVI01_a,NDVI01_b,NDVI02_a,NDVI02_b,NDVI03_a,NDVI03_b,NDVI04_a,NDVI04_b,...,NDVI08_a,NDVI08_b,NDVI09_a,NDVI09_b,NDVI10_a,NDVI10_b,NDVI11_a,NDVI11_b,NDVI12_a,NDVI12_b
0,115.944450,28.696499,0.2072,0.2237,0.2510,0.2677,0.2674,0.2527,0.1888,0.1656,...,0.3762,0.3378,0.3015,0.2851,0.2888,0.2934,0.2869,0.2661,0.2222,0.2211
1,114.523780,26.624906,0.7611,0.7252,0.6900,0.6767,0.6682,0.6885,0.7234,0.7534,...,0.8390,0.8538,0.8333,0.8171,0.7883,0.7604,0.7572,0.7588,0.7712,0.7751
2,114.399671,29.053060,-0.2352,-0.2393,-0.1888,-0.1612,-0.0635,-0.0197,0.0537,0.0455,...,0.0046,0.0460,0.1002,0.1068,0.0982,0.0070,-0.0563,-0.1254,-0.1821,-0.2167
3,114.373410,28.859427,0.7180,0.7039,0.6884,0.6876,0.6930,0.7016,0.7435,0.7749,...,0.8287,0.8274,0.8225,0.8184,0.8113,0.8017,0.7900,0.7740,0.7500,0.7473
4,115.861399,25.148718,0.7371,0.7168,0.6988,0.6934,0.6997,0.7080,0.7444,0.7724,...,0.8350,0.8435,0.8517,0.8493,0.8417,0.8307,0.8177,0.7980,0.7858,0.7710
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4969,116.738013,29.075583,0.2655,0.3314,0.3866,0.3764,0.3455,0.3283,0.3006,0.3220,...,0.6015,0.5811,0.5776,0.5754,0.5427,0.4459,0.3499,0.2420,0.1844,0.1870
4970,116.877634,29.295952,0.6798,0.7060,0.7408,0.7611,0.7618,0.7514,0.7512,0.7507,...,0.8266,0.8187,0.8204,0.8227,0.8154,0.7901,0.7586,0.7173,0.6824,0.6716
4971,116.323362,28.661318,-0.2130,-0.2516,-0.2397,-0.1901,-0.1803,-0.1245,0.0182,0.0890,...,-0.3061,-0.2604,-0.1943,-0.1427,-0.1339,-0.1393,-0.1551,-0.1589,-0.1846,-0.2019
4972,117.726469,28.687242,0.6863,0.5512,0.4851,0.3357,0.4433,0.4233,0.6707,0.7768,...,0.8044,0.8067,0.8095,0.7683,0.6905,0.6520,0.6676,0.7246,0.7619,0.7411


In [12]:
df3.to_csv("E:/Data/temp/Landsat_250.csv")